In [1]:
import pandas as pd
import numpy as np

# Load all datasets
fund_master = pd.read_csv("01_fund_master (1).csv")
nav_history = pd.read_csv("02_nav_history (1).csv")
aum = pd.read_csv("03_aum_by_fund_house (1).csv")
sip = pd.read_csv("04_monthly_sip_inflows (1).csv")
category = pd.read_csv("05_category_inflows (1).csv")
folio = pd.read_csv("06_industry_folio_count (1).csv")
performance = pd.read_csv("07_scheme_performance (1).csv")
transactions = pd.read_csv("08_investor_transactions (1).csv")
portfolio = pd.read_csv("09_portfolio_holdings (1).csv")
benchmark = pd.read_csv("10_benchmark_indices (1).csv")

datasets = {
    "Fund Master": fund_master,
    "NAV History": nav_history,
    "AUM": aum,
    "SIP": sip,
    "Category": category,
    "Folio": folio,
    "Performance": performance,
    "Transactions": transactions,
    "Portfolio": portfolio,
    "Benchmark": benchmark
}

print("="*60)
print("ALL DATASETS LOADED SUCCESSFULLY")
print("="*60)

for name, df in datasets.items():
    print(f"\n{name}")
    print("Rows :", df.shape[0])
    print("Columns :", df.shape[1])
    print(df.head(2))

ALL DATASETS LOADED SUCCESSFULLY

Fund Master
Rows : 40
Columns : 15
   amfi_code       fund_house                                scheme_name  \
0     119551  SBI Mutual Fund  SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund   SBI Bluechip Fund - Direct Plan - Growth   

  category sub_category     plan launch_date      benchmark  \
0   Equity    Large Cap  Regular  2006-02-14  NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01  NIFTY 100 TRI   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \
0               1.54            1.0             500                1000   
1               0.66            1.0             500                1000   

    fund_manager risk_category sebi_category_code  
0  Sohini Andani      Moderate               EC01  
1  Sohini Andani      Moderate               EC01  

NAV History
Rows : 46000
Columns : 3
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04

In [2]:
print("="*60)
print("MISSING VALUES")
print("="*60)

for name, df in datasets.items():
    print("\n", name)
    print(df.isnull().sum())

MISSING VALUES

 Fund Master
amfi_code             0
fund_house            0
scheme_name           0
category              0
sub_category          0
plan                  0
launch_date           0
benchmark             0
expense_ratio_pct     0
exit_load_pct         0
min_sip_amount        0
min_lumpsum_amount    0
fund_manager          0
risk_category         0
sebi_category_code    0
dtype: int64

 NAV History
amfi_code    0
date         0
nav          0
dtype: int64

 AUM
date              0
fund_house        0
aum_lakh_crore    0
aum_crore         0
num_schemes       0
dtype: int64

 SIP
month                         0
sip_inflow_crore              0
active_sip_accounts_crore     0
new_sip_accounts_lakh         0
sip_aum_lakh_crore            0
yoy_growth_pct               12
dtype: int64

 Category
month               0
category            0
net_inflow_crore    0
dtype: int64

 Folio
month                  0
total_folios_crore     0
equity_folios_crore    0
debt_folios_crore      

In [3]:
print("="*60)
print("DUPLICATE RECORDS")
print("="*60)

for name, df in datasets.items():
    print(name, ":", df.duplicated().sum())

DUPLICATE RECORDS
Fund Master : 0
NAV History : 0
AUM : 0
SIP : 0
Category : 0
Folio : 0
Performance : 0
Transactions : 0
Portfolio : 0
Benchmark : 0


In [4]:
import os

os.makedirs("processed", exist_ok=True)

fund_master.to_csv("processed/fund_master_clean.csv", index=False)
nav_history.to_csv("processed/nav_history_clean.csv", index=False)
aum.to_csv("processed/aum_clean.csv", index=False)
sip.to_csv("processed/sip_clean.csv", index=False)
category.to_csv("processed/category_clean.csv", index=False)
folio.to_csv("processed/folio_clean.csv", index=False)
performance.to_csv("processed/performance_clean.csv", index=False)
transactions.to_csv("processed/transactions_clean.csv", index=False)
portfolio.to_csv("processed/portfolio_clean.csv", index=False)
benchmark.to_csv("processed/benchmark_clean.csv", index=False)

print("✅ All cleaned datasets saved successfully.")

✅ All cleaned datasets saved successfully.


In [5]:
# NAV HISTORY CLEANING

print("="*60)
print("NAV HISTORY CLEANING")
print("="*60)

# Convert Date column to datetime
nav_history['date'] = pd.to_datetime(nav_history['date'], errors='coerce')

# Sort by amfi_code and date
nav_history = nav_history.sort_values(['amfi_code','date'])

# Remove duplicates
nav_history = nav_history.drop_duplicates()

# Forward Fill NAV values
nav_history['nav'] = nav_history.groupby('amfi_code')['nav'].ffill()

# Remove invalid NAV
nav_history = nav_history[nav_history['nav'] > 0]

print(nav_history.head())
print(nav_history.info())

NAV HISTORY CLEANING
      amfi_code       date       nav
5750     100016 2022-01-03  520.4608
5751     100016 2022-01-04  515.0971
5752     100016 2022-01-05  521.7239
5753     100016 2022-01-06  515.7880
5754     100016 2022-01-07  515.1639
<class 'pandas.core.frame.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB
None


In [7]:
print("="*60)
print("TRANSACTION CLEANING")
print("="*60)

transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'],
    errors='coerce'
)

transactions['transaction_type'] = (
    transactions['transaction_type']
    .str.strip()
    .str.title()
)

transactions = transactions[
    transactions['transaction_type'].isin(
        ['Sip','Lumpsum','Redemption']
    )
]

transactions = transactions[
    transactions['amount_inr'] > 0
]

print(transactions.head())
print(transactions.info())

TRANSACTION CLEANING
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              Sip        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              Sip         912   
3   INV003436       2024-01-01     118634              Sip        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque 

In [9]:
print("="*60)
print("PERFORMANCE CLEANING")
print("="*60)

performance['expense_ratio_pct'] = pd.to_numeric(
    performance['expense_ratio_pct'],
    errors='coerce'
)

performance = performance[
    (performance['expense_ratio_pct'] >= 0.1) &
    (performance['expense_ratio_pct'] <= 2.5)
]

print(performance.head())
print(performance.info())

PERFORMANCE CLEANING
   amfi_code                                   scheme_name       fund_house  \
0     119551     SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552      SBI Bluechip Fund - Direct Plan - Growth  SBI Mutual Fund   
2     119598    SBI Small Cap Fund - Regular Plan - Growth  SBI Mutual Fund   
3     119599     SBI Small Cap Fund - Direct Plan - Growth  SBI Mutual Fund   
4     119120  SBI Magnum Gilt Fund - Regular Plan - Growth  SBI Mutual Fund   

    category     plan  return_1yr_pct  return_3yr_pct  return_5yr_pct  \
0  Large Cap  Regular           12.42           12.36           14.45   
1  Large Cap   Direct           15.25           11.30           14.23   
2  Small Cap  Regular           24.56           23.39           20.67   
3  Small Cap   Direct           20.59           23.14           21.82   
4       Gilt  Regular            5.34            6.07            5.43   

   benchmark_3yr_pct  alpha  beta  sharpe_ratio  sortino_ratio  \

In [10]:
nav_history.to_csv(
    "processed/nav_history_clean.csv",
    index=False
)

transactions.to_csv(
    "processed/investor_transactions_clean.csv",
    index=False
)

performance.to_csv(
    "processed/scheme_performance_clean.csv",
    index=False
)

print("✅ Cleaned files saved.")

✅ Cleaned files saved.


In [11]:
import sqlite3

# Create Database
conn = sqlite3.connect("bluestock_mf.db")

print("Database Created Successfully")

Database Created Successfully


In [12]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_fund(
    fund_id INTEGER PRIMARY KEY,
    amfi_code INTEGER,
    fund_name TEXT,
    category TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_date(
    date_id INTEGER PRIMARY KEY,
    full_date DATE,
    month INTEGER,
    year INTEGER
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_nav(
    nav_id INTEGER PRIMARY KEY,
    amfi_code INTEGER,
    date DATE,
    nav REAL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_transactions(
    transaction_id INTEGER PRIMARY KEY,
    transaction_date DATE,
    transaction_type TEXT,
    amount REAL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_performance(
    performance_id INTEGER PRIMARY KEY,
    expense_ratio REAL
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_aum(
    aum_id INTEGER PRIMARY KEY,
    fund_house TEXT,
    aum REAL
);
""")

conn.commit()

print("Tables Created Successfully")

Tables Created Successfully


In [13]:
fund_master.to_sql("fund_master", conn, if_exists="replace", index=False)

nav_history.to_sql("nav_history", conn, if_exists="replace", index=False)

aum.to_sql("aum_by_fund_house", conn, if_exists="replace", index=False)

sip.to_sql("monthly_sip_inflows", conn, if_exists="replace", index=False)

category.to_sql("category_inflows", conn, if_exists="replace", index=False)

folio.to_sql("industry_folio_count", conn, if_exists="replace", index=False)

performance.to_sql("scheme_performance", conn, if_exists="replace", index=False)

transactions.to_sql("investor_transactions", conn, if_exists="replace", index=False)

portfolio.to_sql("portfolio_holdings", conn, if_exists="replace", index=False)

benchmark.to_sql("benchmark_indices", conn, if_exists="replace", index=False)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [14]:
tables = [
    "fund_master",
    "nav_history",
    "aum_by_fund_house",
    "monthly_sip_inflows",
    "category_inflows",
    "industry_folio_count",
    "scheme_performance",
    "investor_transactions",
    "portfolio_holdings",
    "benchmark_indices"
]

for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) AS Total FROM {table}", conn)
    print(table)
    print(count)
    print("-"*40)

fund_master
   Total
0     40
----------------------------------------
nav_history
   Total
0  46000
----------------------------------------
aum_by_fund_house
   Total
0     90
----------------------------------------
monthly_sip_inflows
   Total
0     48
----------------------------------------
category_inflows
   Total
0    144
----------------------------------------
industry_folio_count
   Total
0     21
----------------------------------------
scheme_performance
   Total
0     40
----------------------------------------
investor_transactions
   Total
0  32778
----------------------------------------
portfolio_holdings
   Total
0    322
----------------------------------------
benchmark_indices
   Total
0   8050
----------------------------------------


In [16]:
# Query 1
pd.read_sql("""
SELECT * FROM aum_by_fund_house
ORDER BY aum_crore DESC
LIMIT 5;
""", conn)

# Query 2
pd.read_sql("""
SELECT AVG(nav) AS Average_NAV
FROM nav_history;
""", conn)

# Query 3
pd.read_sql("""
SELECT transaction_type,
SUM(amount_inr) AS Total_Amount
FROM investor_transactions
GROUP BY transaction_type;
""", conn)

# Query 4
pd.read_sql("""
SELECT category,
COUNT(*) AS Total_Funds
FROM fund_master
GROUP BY category;
""", conn)

# Query 5
pd.read_sql("""
SELECT *
FROM scheme_performance
ORDER BY expense_ratio_pct ASC;
""", conn)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,118636,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Gilt,Regular,6.19,5.31,8.71,4.42,0.89,0.37,1.33,2.38,4.0,-2.23,30030,0.55,4,Low
1,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
2,120844,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Liquid,Regular,4.26,6.18,8.26,4.66,1.52,0.47,6.18,9.70,0.5,-3.81,27623,0.60,3,Low
3,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
4,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
5,118633,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Large Cap,Direct,14.42,12.33,14.72,10.63,1.70,1.02,0.88,1.48,14.0,-18.14,39475,0.72,3,Moderate
6,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
7,119093,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,Large Cap,Direct,13.95,12.14,13.66,10.71,1.43,0.87,0.87,1.54,14.0,-17.40,15866,0.75,4,Moderate
8,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
9,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High


In [18]:
# Query 6
pd.read_sql("""
SELECT fund_house,
SUM(aum_crore) AS Total_AUM
FROM aum_by_fund_house
GROUP BY fund_house;
""", conn)

# Query 7
pd.read_sql("""
SELECT MAX(nav) AS Highest_NAV
FROM nav_history;
""", conn)

# Query 8
pd.read_sql("""
SELECT MIN(nav) AS Lowest_NAV
FROM nav_history;
""", conn)

# Query 9
pd.read_sql("""
SELECT AVG(expense_ratio_pct) AS Average_Expense
FROM scheme_performance;
""", conn)

# Query 10
pd.read_sql("""
SELECT COUNT(*) AS Total_Transactions
FROM investor_transactions;
""", conn)

,Total_Transactions
0,32778


In [19]:
for name, df in datasets.items():
    print("="*60)
    print(name)
    print(df.columns.tolist())

Fund Master
['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
NAV History
['amfi_code', 'date', 'nav']
AUM
['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
SIP
['month', 'sip_inflow_crore', 'active_sip_accounts_crore', 'new_sip_accounts_lakh', 'sip_aum_lakh_crore', 'yoy_growth_pct']
Category
['month', 'category', 'net_inflow_crore']
Folio
['month', 'total_folios_crore', 'equity_folios_crore', 'debt_folios_crore', 'hybrid_folios_crore', 'others_folios_crore']
Performance
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
Transactions
['inv